In [ ]:
from get_court_position_methods import calibrate_camera, get_3d_position
import cv2
import numpy as np
import torch as t
import torchvision
import torchaudio
import albumentations as A
from ultralytics import YOLO
import matplotlib.pyplot as plt
import mpl_toolkits.mplot3d

In [ ]:
def init_kalman_filter(fps=60.0):
    """
    Ініціалізує 3D фільтр Калмана для трекінгу волейбольного м'яча.
    """
    # 9 динамічних параметрів (x, y, z, vx, vy, vz, ax, ay, az)
    # 3 параметри вимірювання (тільки x, y, z, оскільки швидкість ми не вимірюємо прямо)
    kf = cv2.KalmanFilter(9, 3)

    dt = 1.0 / fps

    kf.transitionMatrix = np.array([
        [1, 0, 0, dt, 0,  0, 0.5*dt**2, 0,         0],
        [0, 1, 0, 0, dt,  0, 0,         0.5*dt**2, 0],
        [0, 0, 1, 0, 0,  dt, 0,         0,         0.5*dt**2],
        [0, 0, 0, 1, 0,  0,  dt,        0,         0],
        [0, 0, 0, 0, 1,  0,  0,         dt,        0],
        [0, 0, 0, 0, 0,  1,  0,         0,         dt],
        [0, 0, 0, 0, 0,  0,  1,         0,         0],
        [0, 0, 0, 0, 0,  0,  0,         1,         0],
        [0, 0, 0, 0, 0,  0,  0,         0,         1]
    ], np.float32)

    kf.measurementMatrix = np.array([
        [1, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0, 0, 0]], np.float32)

    # 3. Коваріація шуму вимірювання (Measurement Noise - R)
    # Наскільки ми довіряємо нашим X, Y, Z.
    # Оскільки Z (глибина) рахується через BBox і має Jitter, ми ставимо для Z більший шум.
    # Значення підбираються емпірично. Чим вище число, тим більше фільтр згладжує.
    kf.measurementNoiseCov = np.array([
        [0.1, 0,   0],
        [0,   0.1, 0],
        [0,   0,   1.0]], np.float32)

    q = np.zeros((9, 9), np.float32)
    np.fill_diagonal(q[0:3, 0:3], 1e-4)
    np.fill_diagonal(q[3:6, 3:6], 0.01)
    np.fill_diagonal(q[6:9, 6:9], .1)

    kf.processNoiseCov = q

    return kf

In [ ]:
main_model = YOLO('/home/var-roman/Desktop/my_projects/diploma_project/training_models/models/main_model_april.pt')

results = main_model.predict(
    source='/home/var-roman/Desktop/my_projects/diploma_project/data/videos/Japan_vs_Poland_ultrashort.mp4',
    imgsz=1920,
    conf=0.4,
    iou=0.45,
    save=False,
    stream=True
)

raw_detections = []
frame_count = 0

for r in results:
    frame_count += 1

    if len(r.boxes) > 0:
        box = r.boxes[0].xywh[0]
        raw_detections.append({
            'ball_detected': True,
            'frame': frame_count,
            'x_pos': float(box[0]),
            'y_pos': float(box[1]),
            'w_box': float(box[2]),
        })
    else:
        raw_detections.append({'ball_detected': False, 'frame': frame_count})

In [ ]:
raw_detections

In [ ]:
pts_real_3d = np.array([
    [0.0, 0.0, 0.0],
    [9.0, 0.0, 0.0],
    [9.0, 18.0, 0.0],
    [0.0, 18.0, 0.0]],dtype=np.float32)

pts_video_2d = np.array([
    [73, 1031],
    [1835, 1027],
    [1504, 606],
    [405, 606]],dtype=np.float32)

K = np.array([
    [1300.0, 0.0,    960.0],
    [0.0,    1300.0, 540.0],
    [0.0,    0.0,    1.0]],dtype=np.float32)

dist_coeffs = np.zeros((4, 1))
R_matrix, t_vec, camera_pos = calibrate_camera(pts_real_3d, pts_video_2d, K, dist_coeffs)

In [ ]:
kf = init_kalman_filter(fps=50.0)
is_initialized = False

smoothed_trajectory = []
speeds_kmh = []

for frame_data in raw_detections:
    prediction = kf.predict()
    pred_x, pred_y, pred_z = prediction[0], prediction[1], prediction[2]


    if frame_data['ball_detected']:
        u_cam, v_cam, w_cam = frame_data['x_pos'], frame_data['y_pos'], frame_data['w_box']
        raw_x, raw_y, raw_z = get_3d_position(u_cam, v_cam, w_cam, K, R_matrix, camera_pos, ball_diameter=0.21)
        measurement = np.array([[raw_x], [raw_y], [raw_z]], dtype=np.float32)

        if not is_initialized:
            kf.statePost = np.array([[raw_x], [raw_y], [raw_z], [0], [0], [0], [0], [0], [-9.8]], dtype=np.float32)
            is_initialized = True
            smoothed_pos = measurement
        else:
            smoothed_pos = kf.correct(measurement)
    else:
        if is_initialized:
            smoothed_pos = prediction
        else:
            continue

    smoothed_trajectory.append([float(smoothed_pos[0][0]), float(smoothed_pos[1][0]), float(smoothed_pos[2][0])])

    vx, vy, vz = kf.statePost[3], kf.statePost[4], kf.statePost[5]
    speed_ms = np.sqrt(vx**2 + vy**2 + vz**2)
    speeds_kmh.append(float(speed_ms[0] * 3.6))

print("Максимальна швидкість спайку:", max(speeds_kmh), "км/год")

In [ ]:
smoothed_trajectory = np.array(smoothed_trajectory)
X_trajectory, Y_trajectory, Z_trajectory = smoothed_trajectory[:, 0], smoothed_trajectory[:, 1], smoothed_trajectory[:, 2]

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = plt.axes(projection='3d')
ax.plot3D(smoothed_trajectory)